# Bayesian Statistics — Runnable Examples

These examples use Python and NumPy. Run the cells from top to bottom. They cover Bayes' theorem, conjugate updates, grid approximation, predictive probabilities, and prior sensitivity.

In [1]:
import math
import numpy as np

print('NumPy version:', np.__version__)

NumPy version: 2.3.5


## 1. Bayes' theorem

A helper function computes a posterior probability from a prior and two conditional probabilities.

In [2]:
def bayes(prior, likelihood_h, likelihood_not_h):
    values = (prior, likelihood_h, likelihood_not_h)
    if any(not 0 <= x <= 1 for x in values):
        raise ValueError("Probabilities must lie in [0, 1].")
    evidence = likelihood_h * prior + likelihood_not_h * (1 - prior)
    if evidence == 0:
        raise ValueError("Observed evidence has zero marginal probability.")
    return likelihood_h * prior / evidence

p = bayes(0.20, 0.80, 0.10)
print(f"Posterior probability: {p:.4f}")

Posterior probability: 0.6667


## 2. Base-rate example

This example illustrates how prevalence and false positives affect the posterior probability after a positive test.

In [3]:
prevalence = 0.01
sensitivity = 0.90
false_positive_rate = 0.05
posterior_positive = bayes(prevalence, sensitivity, false_positive_rate)
print(f"P(condition | positive) = {posterior_positive:.4f}")
print(f"Percentage = {posterior_positive * 100:.2f}%")

P(condition | positive) = 0.1538
Percentage = 15.38%


## 3. Beta-Bernoulli conjugate update

A Beta(α, β) prior updated with successes and failures yields Beta(α + successes, β + failures).

In [4]:
def beta_posterior(alpha, beta, successes, failures):
    if alpha <= 0 or beta <= 0:
        raise ValueError("Prior parameters must be positive.")
    if successes < 0 or failures < 0:
        raise ValueError("Counts must be non-negative.")
    return alpha + successes, beta + failures

a_post, b_post = beta_posterior(1, 1, successes=7, failures=3)
posterior_mean = a_post / (a_post + b_post)
print(f"Posterior: Beta({a_post}, {b_post})")
print(f"Posterior mean: {posterior_mean:.4f}")

Posterior: Beta(8, 4)
Posterior mean: 0.6667


## 4. Posterior predictive probability

For a Beta-Bernoulli model, the predictive probability of success on the next trial is the posterior mean.

In [5]:
def beta_predictive_success(alpha, beta, successes, failures):
    a_post, b_post = beta_posterior(alpha, beta, successes, failures)
    return a_post / (a_post + b_post)

predictive = beta_predictive_success(2, 2, successes=12, failures=8)
print(f"Predictive probability of next success: {predictive:.4f}")

Predictive probability of next success: 0.5833


## 5. Grid approximation

Evaluate an unnormalized posterior over a grid of θ values, then normalize the weights. A uniform prior is constant on this grid.

In [6]:
theta = np.linspace(0.01, 0.99, 500)
successes, failures = 5, 2
weights = theta**successes * (1 - theta)**failures
weights = weights / weights.sum()
grid_mean = np.sum(theta * weights)
grid_map = theta[np.argmax(weights)]
print(f"Weight sum: {weights.sum():.6f}")
print(f"Approximate posterior mean: {grid_mean:.4f}")
print(f"Grid MAP estimate: {grid_map:.4f}")

Weight sum: 1.000000
Approximate posterior mean: 0.6667
Grid MAP estimate: 0.7151


## 6. Posterior simulation and credible interval

Sampling from the posterior approximates summaries such as means and equal-tailed credible intervals.

In [7]:
rng = np.random.default_rng(42)
draws = rng.beta(8, 4, size=100_000)
ci_low, ci_high = np.quantile(draws, [0.025, 0.975])
print(f"Simulated mean: {draws.mean():.4f}")
print(f"Exact mean: {8 / 12:.4f}")
print(f"Approximate 95% credible interval: ({ci_low:.4f}, {ci_high:.4f})")

Simulated mean: 0.6671
Exact mean: 0.6667
Approximate 95% credible interval: (0.3939, 0.8909)


## 7. Prior sensitivity

Compare posterior means under several prior choices for the same observations.

In [8]:
observed_successes, observed_failures = 7, 3
priors = [(1, 1), (2, 2), (10, 10)]
for alpha, beta in priors:
    ap, bp = beta_posterior(alpha, beta, observed_successes, observed_failures)
    mean = ap / (ap + bp)
    print(f"Prior Beta({alpha}, {beta}) -> Posterior Beta({ap}, {bp}), mean={mean:.4f}")

Prior Beta(1, 1) -> Posterior Beta(8, 4), mean=0.6667
Prior Beta(2, 2) -> Posterior Beta(9, 5), mean=0.6429
Prior Beta(10, 10) -> Posterior Beta(17, 13), mean=0.5667


## 8. Sanity checks

Small assertions help catch implementation errors.

In [9]:
assert abs(bayes(0.20, 0.80, 0.10) - (0.16 / 0.24)) < 1e-12
assert beta_posterior(1, 1, 7, 3) == (8, 4)
assert abs(beta_predictive_success(2, 2, 12, 8) - (14 / 24)) < 1e-12
assert abs(weights.sum() - 1.0) < 1e-12
print("All sanity checks passed.")

All sanity checks passed.
